# Atlas-100M-TXL: a 20.5M-parameter traffic share on the frozen Atlas-100M (ParadoxAI Lab)

Rules: `atlas/m100/PREREG_TRAFFIC.md`, addendum B, committed before this ran. Weather and air quality are not touched (check W). A second Colab session; runs alongside `atlas_100m_traffic_plus_colab.ipynb`. A100 → Run all → allow Google Drive (the account holding `PWM-100M/`). Resumes from Drive.

In [ ]:
# 1. Drive and GPU
import os, shutil, hashlib, json, glob
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)
STORE = '/content/drive/MyDrive/PWM-100M'; OUT = f'{STORE}/out'; TP = f'{STORE}/traffic_plus'
assert os.path.exists(f'{OUT}/atlas100m_traffic_view.pt'), 'Atlas-100M traffic view missing'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Code and data (traffic only); the published submissions only (Holm family fixed by addendum B)
!rm -rf /content/pwm && git clone -q --depth 1 -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model.git /content/pwm
!cd /content/pwm && git log --oneline -1
!pip install -q einops 2>&1 | tail -1
V06 = '/content/pwm/v0.6-conference'
!cd {V06} && python tools/fetch_all.py 2>&1 | tail -4
for d in ('submissions', 'results/paradoxbench'):
    shutil.copytree(f'{STORE}/v06/{d}', f'{V06}/{d}', dirs_exist_ok=True)
for f in glob.glob(f'{V06}/submissions/*Atlas-100M-T*') + glob.glob(f'{V06}/results/paradoxbench/*Atlas-100M-T*'):
    os.remove(f)                                                   # T / T1d are not in TXL's family (addendum B)
print(len(glob.glob(f'{V06}/submissions/*.npz')), 'submissions restored')

In [ ]:
# 3. Check W (before)
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()
SUMS = dict(l.split()[::-1] for l in open('/content/pwm/atlas/m100/results/SHA256SUMS'))
W = {f: sha(f'{OUT}/{f}') == SUMS[f] for f in ['atlas100m_best.pt', 'atlas100m_weather_view.pt', 'atlas100m_aq_view.pt']}
print('CHECK W before:', W); assert all(W.values())

In [ ]:
# 4. Train Atlas-100M-TXL (check T0 inside; resumes from Drive)
!cd {V06} && python -u ../atlas/m100/traffic_plus.py train --base {OUT}/atlas100m_traffic_view.pt --out {TP} --r 128 --alpha 128 --E 64 --ctx 288 --steps 60000 --cap 360 --name Atlas-100M-TXL --tag atlas100mTXL 2>&1 | grep --line-buffered -v "Warning\|warn(" 

In [ ]:
# 5. Score once (Track B), save, check W (after), copy results
!cd {V06} && python -u ../atlas/m100/score_traffic.py --plus {TP}/atlas100mTXL_best.pt --name Atlas-100M-TXL --summary {OUT}/atlas100m_summary.json --out {TP} 2>&1 | grep --line-buffered "ParadoxBench\|TEST\|Error"
for f in glob.glob(f'{V06}/submissions/*Atlas-100M-TXL*') + glob.glob(f'{V06}/results/paradoxbench/*Atlas-100M-TXL*'):
    shutil.copy(f, f'{STORE}/v06/' + ('submissions' if f.endswith('.npz') else 'results/paradoxbench'))
W = {f: sha(f'{OUT}/{f}') == SUMS[f] for f in ['atlas100m_best.pt', 'atlas100m_weather_view.pt', 'atlas100m_aq_view.pt']}
print('CHECK W after:', W)
DST = '/content/drive/MyDrive/PWM-10M/m100_out'; os.makedirs(DST, exist_ok=True)
for f in glob.glob(f'{TP}/*TXL*.json') + glob.glob(f'{TP}/*TXL*.jsonl') + glob.glob(f'{TP}/atlas100mtxl*.json'): shutil.copy(f, DST)
print(sorted(os.listdir(DST)))